# sfig4 — Iso-Compute Heatmap, Additional Tasks (Supplementary Fig. S-4)

Full (L, K) heatmap with iso-compute contours for tasks not shown in main paper Fig. 3.
Panels: (a) age, (b) depression, (c) OSA.

**Data**: `heatmap_df_test.csv` per task, Transformer head.

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root ─��──────────────────────────────────────────────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Main utils (TBME style, data, panels)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS,
    TASK_LABEL, FONT_BASE, FONT_LABEL, CTX_ORDER,
)
from utils.data import set_root, load_analysis, load_analysis_all_k, load_heatmap, load_parquets
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
HEAD  = "transformer"
SPLIT = "test"
TASKS = ["age_class", "depression_extreme_binary", "osa_binary_apples_postqc"]

hmaps = {t: load_heatmap("phase0_v3", t, HEAD) for t in TASKS}
print("Heatmap sizes:", {t: len(v) for t, v in hmaps.items()})

In [ ]:
METRIC      = "auroc"
SUPP_NCOLS  = 2
SUPP_ROW_H  = 2.0
SUPP_NROWS  = (len(TASKS) + SUPP_NCOLS - 1) // SUPP_NCOLS

# 2+1 centred mosaic (each label occupies 2 virtual columns)
_sl = [chr(97 + _i) for _i in range(len(TASKS))]
_n_last = len(TASKS) % SUPP_NCOLS or SUPP_NCOLS
_n_full  = len(TASKS) // SUPP_NCOLS
_mosaic  = []
for _r in range(_n_full):
    _row = _sl[_r * SUPP_NCOLS : (_r + 1) * SUPP_NCOLS]
    _mosaic.append([lbl for lbl in _row for _ in range(2)])
if _n_last < SUPP_NCOLS:
    _last = _sl[_n_full * SUPP_NCOLS:]
    _pad  = SUPP_NCOLS - _n_last
    _mosaic.append(['.'] * _pad +
                   [lbl for lbl in _last for _ in range(2)] +
                   ['.'] * _pad)

fig, axd = plt.subplot_mosaic(
    _mosaic, figsize=(FULL_W - 2, SUPP_NROWS * SUPP_ROW_H)
)

# Left col: y-labels on, cbar unlabelled; Right col: y-labels off, cbar labelled
_supp_flags = [(True, False), (False, True), (True, False)]
for _i, (_lbl, _task) in enumerate(zip(_sl, TASKS)):
    _sy, _sc = _supp_flags[_i]
    panels.heatmap_panel(axd[_lbl], hmaps[_task], col=METRIC,
                         show_ylabels=_sy, show_cbar_label=_sc)
    axd[_lbl].set_title(TASK_LABEL[_task], fontsize=8, pad=3)
    add_panel_label(axd[_lbl], f'({chr(97+_i)})')

fig.tight_layout(h_pad=0.5, w_pad=-2.0)
plt.show()

In [ ]:
save_figure(fig, FINAL_OUT, "sfig4_heatmap")
import shutil
shutil.copy(FINAL_OUT / "sfig4_heatmap.pdf",
            WORKSPACE_ROOT / "TBME_submission" / "sfig4_heatmap.pdf")
print("Saved + copied → TBME_submission/sfig4_heatmap.pdf")